# Efficient evaluation of expectation

In [2]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp, Pauli
from typing import Union, List

def expectation_value_all_zeros(observable: Union[SparsePauliOp, List], n_qubits: int = None) -> complex:
    """
    Compute the expectation value of an observable with respect to the all-zeros state |0^n⟩.
    
    The algorithm uses the fact that the Pauli decomposition of ρ_0 = |0^n⟩⟨0^n| 
    contains only Pauli strings with I and Z operators, each with coefficient 1/2^n.
    
    Args:
        observable: Either a SparsePauliOp object or a list of (pauli_string, coefficient) tuples
        n_qubits: Number of qubits (required if observable is a list)
    
    Returns:
        The expectation value Tr(ρ_0 O)
    """
    
    # Convert to SparsePauliOp if necessary
    if isinstance(observable, list):
        if n_qubits is None:
            raise ValueError("n_qubits must be specified when observable is a list")
        pauli_strings = [item[0] for item in observable]
        coefficients = [item[1] for item in observable]
        observable = SparsePauliOp(pauli_strings, coefficients)
    
    # Get the number of qubits
    n = observable.num_qubits
    
    # Initialize expectation value
    expectation = 0.0
    
    # Iterate through all Pauli terms in the observable
    for pauli_string, coeff in zip(observable.paulis, observable.coeffs):
        # Convert Pauli object to string representation
        pauli_str = str(pauli_string)
        
        # Check if this Pauli string contains only I and Z
        # (no X or Y operators)
        if all(op in ['I', 'Z'] for op in pauli_str):
            # This term contributes to the expectation value
            expectation += coeff
    
    return expectation


def create_all_zeros_density_matrix(n_qubits: int) -> SparsePauliOp:
    """
    Create the Pauli decomposition of the all-zeros density matrix ρ_0 = |0^n⟩⟨0^n|.
    
    Note: This function is for verification purposes. For large n, it creates 2^n terms,
    so use with caution for n > 10.
    
    Args:
        n_qubits: Number of qubits
    
    Returns:
        SparsePauliOp representing ρ_0 in Pauli basis
    """
    from itertools import product
    
    pauli_strings = []
    coefficients = []
    
    # Generate all combinations of I and Z
    for ops in product(['I', 'Z'], repeat=n_qubits):
        pauli_str = ''.join(ops)
        pauli_strings.append(pauli_str)
        coefficients.append(1.0 / (2**n_qubits))
    
    return SparsePauliOp(pauli_strings, coefficients)


def verify_expectation_value(observable: SparsePauliOp, n_qubits: int = None) -> dict:
    """
    Verify the expectation value calculation by also computing it using the state vector.
    
    Args:
        observable: SparsePauliOp object
        n_qubits: Number of qubits (uses observable.num_qubits if not specified)
    
    Returns:
        Dictionary with both calculation methods and their results
    """
    from qiskit.quantum_info import Statevector
    
    if n_qubits is None:
        n_qubits = observable.num_qubits
    
    # Method 1: Efficient Pauli basis calculation
    exp_val_pauli = expectation_value_all_zeros(observable)
    
    # Method 2: Direct calculation using state vector
    all_zeros_state = Statevector.from_label('0' * n_qubits)
    exp_val_direct = all_zeros_state.expectation_value(observable).real
    
    return {
        'pauli_method': exp_val_pauli,
        'direct_method': exp_val_direct,
        'difference': abs(exp_val_pauli - exp_val_direct)
    }


In [3]:
# Example usage and demonstration
if __name__ == "__main__":
    print("=== Pauli Basis Expectation Value Calculator ===\n")
    
    # Example 1: Simple 2-qubit observable
    print("Example 1: 2-qubit system")
    print("-" * 40)
    
    # Create an observable with multiple Pauli terms
    # O = 0.5*II + 0.3*ZI + 0.2*IZ + 0.1*ZZ + 0.4*XX + 0.2*XY
    pauli_list = [
        ('II', 0.5),
        ('ZI', 0.3),
        ('IZ', 0.2),
        ('ZZ', 0.1),
        ('XX', 0.4),  # This won't contribute (contains X)
        ('XY', 0.2)   # This won't contribute (contains X and Y)
    ]
    
    observable = SparsePauliOp.from_list(pauli_list)
    
    print(f"Observable O = {observable}")
    print()
    
    # Calculate expectation value
    exp_val = expectation_value_all_zeros(observable)
    print(f"Expectation value ⟨0^n|O|0^n⟩ = {exp_val}")
    print(f"Expected: 0.5 + 0.3 + 0.2 + 0.1 = {0.5 + 0.3 + 0.2 + 0.1}")
    print()
    
    # Verify with direct calculation
    verification = verify_expectation_value(observable)
    print("Verification:")
    print(f"  Pauli method: {verification['pauli_method']}")
    print(f"  Direct method: {verification['direct_method']}")
    print(f"  Difference: {verification['difference']:.2e}")
    print()
    
    # Example 2: Random 3-qubit observable
    print("\nExample 2: Random 3-qubit observable")
    print("-" * 40)
    
    # Generate a random observable
    np.random.seed(42)
    n_terms = 10
    pauli_strings = []
    coeffs = []
    
    for _ in range(n_terms):
        # Random Pauli string
        ops = np.random.choice(['I', 'X', 'Y', 'Z'], size=3)
        pauli_str = ''.join(ops)
        pauli_strings.append(pauli_str)
        
        # Random coefficient
        coeff = np.random.randn() + 1j * np.random.randn()
        coeffs.append(coeff)
    
    observable_3q = SparsePauliOp(pauli_strings, coeffs)
    
    print(f"Number of terms in observable: {len(observable_3q)}")
    
    # Find terms that contribute (only I and Z)
    contributing_terms = []
    for p, c in zip(observable_3q.paulis, observable_3q.coeffs):
        p_str = str(p)
        if all(op in ['I', 'Z'] for op in p_str):
            contributing_terms.append((p_str, c))
    
    print(f"Contributing terms (only I and Z): {len(contributing_terms)}")
    for term, coeff in contributing_terms:
        print(f"  {term}: {coeff}")
    
    # Calculate expectation value
    exp_val_3q = expectation_value_all_zeros(observable_3q)
    print(f"\nExpectation value: {exp_val_3q}")
    
    # Verify
    verification_3q = verify_expectation_value(observable_3q)
    print(f"Verification difference: {verification_3q['difference']:.2e}")
    
    # Example 3: Demonstrating the density matrix decomposition (small system)
    print("\nExample 3: Density matrix decomposition (2 qubits)")
    print("-" * 40)
    
    rho_0 = create_all_zeros_density_matrix(2)
    print(f"ρ_0 = |00⟩⟨00| in Pauli basis:")
    print(f"Number of terms: {len(rho_0)} (should be 2^2 = 4)")
    print("\nTerms:")
    for p, c in zip(rho_0.paulis, rho_0.coeffs):
        print(f"  {str(p)}: {c}")
    
    print("\n=== Done ===")

=== Pauli Basis Expectation Value Calculator ===

Example 1: 2-qubit system
----------------------------------------
Observable O = SparsePauliOp(['II', 'ZI', 'IZ', 'ZZ', 'XX', 'XY'],
              coeffs=[0.5+0.j, 0.3+0.j, 0.2+0.j, 0.1+0.j, 0.4+0.j, 0.2+0.j])

Expectation value ⟨0^n|O|0^n⟩ = (1.1+0j)
Expected: 0.5 + 0.3 + 0.2 + 0.1 = 1.1

Verification:
  Pauli method: (1.1+0j)
  Direct method: 1.1
  Difference: 0.00e+00


Example 2: Random 3-qubit observable
----------------------------------------
Number of terms in observable: 10
Contributing terms (only I and Z): 1
  ZZZ: (-0.7033438017074073-2.1396206560762394j)

Expectation value: (-0.7033438017074073-2.1396206560762394j)
Verification difference: 2.14e+00

Example 3: Density matrix decomposition (2 qubits)
----------------------------------------
ρ_0 = |00⟩⟨00| in Pauli basis:
Number of terms: 4 (should be 2^2 = 4)

Terms:
  II: (0.25+0j)
  IZ: (0.25+0j)
  ZI: (0.25+0j)
  ZZ: (0.25+0j)

=== Done ===


In [8]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp, Statevector
from typing import Union, List, Tuple

def expectation_value_alternating_state(observable: Union[SparsePauliOp, List], 
                                       n_qubits: int = None,
                                       pattern: str = None) -> complex:
    """
    Compute the expectation value of an observable with respect to an alternating state.
    Default pattern is |0101...01⟩ for even number of qubits.
    
    The algorithm uses the Pauli decomposition of the density matrix:
    - Only Pauli strings with I and Z contribute
    - The coefficient depends on the number of Z operators at even positions
    
    Args:
        observable: Either a SparsePauliOp object or a list of (pauli_string, coefficient) tuples
        n_qubits: Number of qubits (required if observable is a list)
        pattern: Optional custom pattern string (e.g., "0101" or "010101")
    
    Returns:
        The expectation value Tr(ρ O)
    """
    
    # Convert to SparsePauliOp if necessary
    if isinstance(observable, list):
        if n_qubits is None:
            raise ValueError("n_qubits must be specified when observable is a list")
        pauli_strings = [item[0] for item in observable]
        coefficients = [item[1] for item in observable]
        observable = SparsePauliOp(pauli_strings, coefficients)
    
    # Get the number of qubits
    n = observable.num_qubits
    
    # Determine the pattern
    if pattern is None:
        if n % 2 != 0:
            raise ValueError("Default pattern requires even number of qubits")
        # Create alternating pattern "0101...01"
        pattern = '01' * (n // 2)
    else:
        if len(pattern) != n:
            raise ValueError(f"Pattern length {len(pattern)} doesn't match number of qubits {n}")
    
    # Initialize expectation value
    expectation = 0.0
    
    # Iterate through all Pauli terms in the observable
    for pauli_string, coeff in zip(observable.paulis, observable.coeffs):
        # Convert Pauli object to string representation
        pauli_str = str(pauli_string)
        
        # Check if this Pauli string contains only I and Z
        if all(op in ['I', 'Z'] for op in pauli_str):
            # Count Z operators at positions where pattern has '1'
            sign_exponent = 0
            for i, (op, bit) in enumerate(zip(pauli_str, pattern)):
                if op == 'Z' and bit == '1':
                    sign_exponent += 1
            
            # This term contributes with appropriate sign
            contribution = coeff * ((-1) ** sign_exponent)
            expectation += contribution
    
    return expectation


def create_alternating_density_matrix(n_qubits: int, pattern: str = None) -> SparsePauliOp:
    """
    Create the Pauli decomposition of an alternating state density matrix.
    
    Note: This function creates 2^n terms, so use with caution for n > 10.
    
    Args:
        n_qubits: Number of qubits (must be even for default pattern)
        pattern: Optional pattern string (e.g., "0101")
    
    Returns:
        SparsePauliOp representing ρ in Pauli basis
    """
    from itertools import product
    
    if pattern is None:
        if n_qubits % 2 != 0:
            raise ValueError("Default pattern requires even number of qubits")
        pattern = '01' * (n_qubits // 2)
    else:
        if len(pattern) != n_qubits:
            raise ValueError(f"Pattern length doesn't match n_qubits")
    
    pauli_strings = []
    coefficients = []
    
    # Generate all combinations of I and Z
    for ops in product(['I', 'Z'], repeat=n_qubits):
        pauli_str = ''.join(ops)
        
        # Calculate coefficient based on pattern
        sign_exponent = 0
        for i, (op, bit) in enumerate(zip(ops, pattern)):
            if op == 'Z' and bit == '1':
                sign_exponent += 1
        
        coeff = ((-1) ** sign_exponent) / (2 ** n_qubits)
        
        pauli_strings.append(pauli_str)
        coefficients.append(coeff)
    
    return SparsePauliOp(pauli_strings, coefficients)


def verify_expectation_alternating(observable: SparsePauliOp, 
                                  n_qubits: int = None,
                                  pattern: str = None) -> dict:
    """
    Verify the expectation value calculation using direct state vector method.
    
    Args:
        observable: SparsePauliOp object
        n_qubits: Number of qubits
        pattern: State pattern (default "0101...01")
    
    Returns:
        Dictionary with both calculation methods and their results
    """
    if n_qubits is None:
        n_qubits = observable.num_qubits
    
    if pattern is None:
        if n_qubits % 2 != 0:
            raise ValueError("Default pattern requires even number of qubits")
        pattern = '01' * (n_qubits // 2)
    
    # Method 1: Efficient Pauli basis calculation
    exp_val_pauli = expectation_value_alternating_state(observable, pattern=pattern)
    
    # Method 2: Direct calculation using state vector
    state = Statevector.from_label(pattern)
    exp_val_direct = state.expectation_value(observable).real
    
    return {
        'pattern': pattern,
        'pauli_method': exp_val_pauli,
        'direct_method': exp_val_direct,
        'difference': abs(exp_val_pauli - exp_val_direct)
    }


def analyze_observable_contributions(observable: SparsePauliOp, pattern: str) -> dict:
    """
    Analyze which terms in an observable contribute to the expectation value.
    
    Args:
        observable: SparsePauliOp object
        pattern: State pattern string
    
    Returns:
        Dictionary with analysis results
    """
    contributing_terms = []
    non_contributing_terms = []
    
    for pauli_string, coeff in zip(observable.paulis, observable.coeffs):
        pauli_str = str(pauli_string)
        
        if all(op in ['I', 'Z'] for op in pauli_str):
            # Calculate sign based on Z positions
            sign_exponent = sum(1 for i, (op, bit) in enumerate(zip(pauli_str, pattern)) 
                              if op == 'Z' and bit == '1')
            sign = (-1) ** sign_exponent
            contribution = coeff * sign
            contributing_terms.append({
                'pauli': pauli_str,
                'coeff': coeff,
                'sign': sign,
                'contribution': contribution
            })
        else:
            non_contributing_terms.append({
                'pauli': pauli_str,
                'coeff': coeff
            })
    
    total_contribution = sum(term['contribution'] for term in contributing_terms)
    
    return {
        'contributing_terms': contributing_terms,
        'non_contributing_terms': non_contributing_terms,
        'total_contribution': total_contribution,
        'n_contributing': len(contributing_terms),
        'n_non_contributing': len(non_contributing_terms)
    }


# Example usage and demonstration
if __name__ == "__main__":
    print("=== Pauli Basis Expectation Value for Alternating State ===\n")
    
    # Example 1: 4-qubit alternating state |0101⟩
    print("Example 1: 4-qubit alternating state |0101⟩")
    print("-" * 50)
    
    # Create an observable
    pauli_list_4q = [
        ('IIII', 1.0),    # contributes with +1
        ('ZIIZ', 0.5),    # Z at positions 1,3 (odd) → sign = (-1)^0 = +1
        ('IZZI', 0.3),    # Z at positions 2,3 (1 even, 1 odd) → sign = (-1)^1 = -1
        ('ZZZZ', 0.2),    # Z at all positions (2 even) → sign = (-1)^2 = +1
        ('XIXI', 0.4),    # doesn't contribute (contains X)
        ('YYII', 0.1)     # doesn't contribute (contains Y)
    ]
    
    observable_4q = SparsePauliOp.from_list(pauli_list_4q)
    pattern_4q = '0101'
    
    print(f"State pattern: |{pattern_4q}⟩")
    print(f"Observable: {observable_4q}\n")
    
    # Analyze contributions
    analysis = analyze_observable_contributions(observable_4q, pattern_4q)
    
    print("Contributing terms (only I and Z):")
    for term in analysis['contributing_terms']:
        print(f"  {term['pauli']}: coeff={term['coeff']:.2f}, "
              f"sign={term['sign']:+d}, contribution={term['contribution']:.2f}")
    
    print(f"\nExpected value calculation:")
    print(f"  1.0×(+1) + 0.5×(+1) + 0.3×(-1) + 0.2×(+1) = {1.0 + 0.5 - 0.3 + 0.2}")
    
    # Calculate and verify
    exp_val = expectation_value_alternating_state(observable_4q, pattern=pattern_4q)
    print(f"\nComputed expectation value: {exp_val}")
    
    verification = verify_expectation_alternating(observable_4q, pattern=pattern_4q)
    print(f"Verification with direct method: {verification['direct_method']}")
    print(f"Difference: {verification['difference']:.2e}\n")
    
    # Example 2: 6-qubit alternating state |010101⟩
    print("\nExample 2: 6-qubit alternating state |010101⟩")
    print("-" * 50)
    
    # Generate random observable
    np.random.seed(123)
    n_terms = 15
    pauli_strings_6q = []
    coeffs_6q = []
    
    for _ in range(n_terms):
        ops = np.random.choice(['I', 'X', 'Y', 'Z'], size=6)
        pauli_str = ''.join(ops)
        pauli_strings_6q.append(pauli_str)
        coeffs_6q.append(np.random.randn())
    
    observable_6q = SparsePauliOp(pauli_strings_6q, coeffs_6q)
    pattern_6q = '010101'
    
    print(f"State pattern: |{pattern_6q}⟩")
    print(f"Number of terms in observable: {len(observable_6q)}")
    
    # Analyze
    analysis_6q = analyze_observable_contributions(observable_6q, pattern_6q)
    print(f"Contributing terms: {analysis_6q['n_contributing']}")
    print(f"Non-contributing terms: {analysis_6q['n_non_contributing']}")
    
    # Show some contributing terms
    print("\nSample contributing terms:")
    for term in analysis_6q['contributing_terms'][:5]:
        print(f"  {term['pauli']}: contribution = {term['coeff']:.3f} × {term['sign']:+d} = {term['contribution']:.3f}")
    
    # Calculate and verify
    exp_val_6q = expectation_value_alternating_state(observable_6q, pattern=pattern_6q)
    verification_6q = verify_expectation_alternating(observable_6q, pattern=pattern_6q)
    
    print(f"\nExpectation value: {exp_val_6q:.6f}")
    print(f"Verification difference: {verification_6q['difference']:.2e}\n")
    
    # Example 3: Custom pattern
    print("\nExample 3: Custom pattern |001011⟩")
    print("-" * 50)
    
    custom_pattern = '001011'
    observable_custom = SparsePauliOp.from_list([
        ('IIIIII', 1.0),
        ('ZZIIII', 0.5),
        ('IIIZZZ', 0.3),
        ('ZZZZZZ', 0.2)
    ])
    
    print(f"Custom pattern: |{custom_pattern}⟩")
    analysis_custom = analyze_observable_contributions(observable_custom, custom_pattern)
    
    print("Term contributions:")
    for term in analysis_custom['contributing_terms']:
        z_at_ones = sum(1 for i, (op, bit) in enumerate(zip(term['pauli'], custom_pattern)) 
                       if op == 'Z' and bit == '1')
        print(f"  {term['pauli']}: {term['coeff']:.1f} × (-1)^{z_at_ones} = {term['contribution']:.1f}")
    
    exp_val_custom = expectation_value_alternating_state(observable_custom, pattern=custom_pattern)
    print(f"\nTotal expectation value: {exp_val_custom}")
    
    # Example 4: Density matrix decomposition (small system)
    print("\nExample 4: Density matrix Pauli decomposition for |01⟩")
    print("-" * 50)
    
    rho_01 = create_alternating_density_matrix(2, pattern='01')
    print("ρ = |01⟩⟨01| in Pauli basis:")
    for p, c in zip(rho_01.paulis, rho_01.coeffs):
        if abs(c) > 1e-10:  # Only show non-zero terms
            print(f"  {str(p)}: {c:+.3f}")
    
    print("\n=== Done ===")

=== Pauli Basis Expectation Value for Alternating State ===

Example 1: 4-qubit alternating state |0101⟩
--------------------------------------------------
State pattern: |0101⟩
Observable: SparsePauliOp(['IIII', 'ZIIZ', 'IZZI', 'ZZZZ', 'XIXI', 'YYII'],
              coeffs=[1. +0.j, 0.5+0.j, 0.3+0.j, 0.2+0.j, 0.4+0.j, 0.1+0.j])

Contributing terms (only I and Z):
  IIII: coeff=1.00+0.00j, sign=+1, contribution=1.00+0.00j
  ZIIZ: coeff=0.50+0.00j, sign=-1, contribution=-0.50+0.00j
  IZZI: coeff=0.30+0.00j, sign=-1, contribution=-0.30+0.00j
  ZZZZ: coeff=0.20+0.00j, sign=+1, contribution=0.20+0.00j

Expected value calculation:
  1.0×(+1) + 0.5×(+1) + 0.3×(-1) + 0.2×(+1) = 1.4

Computed expectation value: (0.4+0j)
Verification with direct method: 0.4
Difference: 0.00e+00


Example 2: 6-qubit alternating state |010101⟩
--------------------------------------------------
State pattern: |010101⟩
Number of terms in observable: 15
Contributing terms: 0
Non-contributing terms: 15

Sample contri